# Experimental Codes

In [1]:
from TableTennisEnvironmentV0.TableTennisEnvironment import TableTennisEnv
from utils.math_solvers import solve_quadratic
import pybullet as p 

g = 9.8
env = TableTennisEnv(show_gui=False)

pybullet build time: Feb  4 2024 12:55:26


In [2]:
## We need to estimate the initial velocity of the ball 
## although we can get it in pybullet, we won't get it directly in optitrack

def estimateInitVelocity(ball_position1, ball_position2, time_gap):
    displacement = (ball_position2[0] - ball_position1[0], ball_position2[1] - ball_position1[1], ball_position2[2] - ball_position1[2])
    velocity = [displacement[i] / time_gap for i in range(len(displacement))]
    return velocity
    

In [3]:
## This is real initial velocity
u_x, u_y, u_z = 5.5, 0.8, -3     # in meters/sec
real_initial_velocity = [u_x, u_y, u_z]
env.throw_ball(real_initial_velocity)

## now we estimate the initial velocity
ball_position1 = p.getBasePositionAndOrientation(env._ball)[0]
action = [0, 0, 0, 0, 0, 0]
env.step(action)
ball_position2 = p.getBasePositionAndOrientation(env._ball)[0]
time_gap = 1/240

estimated_initial_velocity = estimateInitVelocity(ball_position1, ball_position2, time_gap)

print('real velocity: ', real_initial_velocity)
print('estimated velocity: ', estimated_initial_velocity)



real velocity:  [5.5, 0.8, -3]
estimated velocity:  [5.493293801502528, 0.7990245529458223, -3.0372170735468096]


So the results are pretty accurate

![Alt text](illustration.jpeg)


In [4]:
# dimensions of the table
[Lx, Ly, Lz] = [2.74, 1.5, 1.0]


# distance between ball and the robot
ball_initial_position = p.getBasePositionAndOrientation(env._ball)[0] # can be collected from optitrack
robot_initial_position = p.getLinkState(env._robotic_arm, 0)[0]      # can be collected from optitrack
distance_x, distance_y, distance_z = (robot_initial_position[0] - ball_initial_position[0], robot_initial_position[1] - ball_initial_position[1], robot_initial_position[2] - ball_initial_position[2])

# height of the ball from the table
h1 = ball_initial_position[2] - Lz - 0.2 # 0.2m is the height of the red surface


# now calculate the time for the contact with ground
g = 9.8
a = -0.5*g
b = estimated_initial_velocity[2]
c = h1

t1 = None # it is the time for the ball to hit the ground for the first time

roots = solve_quadratic(a, b, c)
for root in roots:
    if root.imag == 0 and root.real > 0: # means complex root, ignore
        t1 = root.real


## now how much distance did the 
d1_x = estimated_initial_velocity[0]*t1
d1_y = estimated_initial_velocity[1]*t1


## Velocities at this position
vx = estimated_initial_velocity[0]
vy = estimated_initial_velocity[1]
vz = estimated_initial_velocity[2] - g*t1


## remaining distance that needs to be covered to reach the robot - maybe we need to set constraints on y displacement as well
d2_x = distance_x - d1_x

new_velocity = [vx, vy, -vz] # vz will change sign because now the direction of the projectile in z axis is reversed


## not calculate how high the projectile can go 
t2 = distance_x / estimated_initial_velocity[0] - t1 # remaining flight time to reach the robot
h2 = new_velocity[2]*t2 - 0.5*g*(t2**2)


# now recompute the position of the ball on the Y-Z plane

y = ball_initial_position[1] + estimated_initial_velocity[1] * (t1 + t2)
z = h2 + Lz + 0.2

estimated_hitting_point = (y, z)
print('total flight time: ', t1+t2, 's')
print('The ball will hit the Y-Z plane at: ', estimated_hitting_point)

total flight time:  0.487433502117562 s
The ball will hit the Y-Z plane at:  (0.3927974256909091, 2.22927586372358)


In [5]:
def calculate_flight_times(ball_initial_position, robot_initial_position, estimated_initial_velocity, table_height): # this will return the two flight times - t1 and t2
    distance_x, distance_y, distance_z = (robot_initial_position[0] - ball_initial_position[0], robot_initial_position[1] - ball_initial_position[1], robot_initial_position[2] - ball_initial_position[2])

    # height of the ball from the table
    h1 = ball_initial_position[2] - table_height - 0.2 # 0.2m is the height of the red surface


    # now calculate the time for the contact with ground
    a = -0.5*g
    b = estimated_initial_velocity[2]
    c = h1

    t1 = None # it is the time for the ball to hit the ground for the first time

    roots = solve_quadratic(a, b, c)
    for root in roots:
        if root.imag == 0 and root.real > 0: # means complex root, ignore
            t1 = root.real
    ## not calculate how high the projectile can go 
    t2 = distance_x / estimated_initial_velocity[0] - t1 # remaining flight time to reach the robot

    return t1, t2


# print(calculate_flight_times(ball_initial_position, robot_initial_position, estimated_initial_velocity, Lz))



In [6]:
def calculate_h2(estimated_initial_velocity, t1, t2): # this will provide the second height that the ball will reach: h2
    ## Velocities after bounce
    vx = estimated_initial_velocity[0]
    vy = estimated_initial_velocity[1]
    vz = estimated_initial_velocity[2] - g*t1
    new_velocity = [vx, vy, -vz] # vz will change sign because now the direction of the projectile in z axis is reversed
    h2 = new_velocity[2]*t2 - 0.5*g*(t2**2)
    return h2
    

In [7]:
def estimate_hitting_point(ball_initial_position, robot_initial_position, estimated_initial_velocity, table_height, base_surface_thickness = 0.2): # this will return the estimated hitting point in the Y-Z plane
    t1, t2 = calculate_flight_times(ball_initial_position, robot_initial_position, estimated_initial_velocity, table_height)
    h2 = calculate_h2(estimated_initial_velocity, t1, t2)

    y = ball_initial_position[1] + estimated_initial_velocity[1] * (t1 + t2)
    z = h2 + table_height + base_surface_thickness
    return (y, z)
## get a new ball
env.get_new_ball(position=[-1, 0.1, 2])


## Throw the ball
u_x, u_y, u_z = 5.5, 0.8, -3     # in meters/sec
real_initial_velocity = [u_x, u_y, u_z]
env.throw_ball(real_initial_velocity)


ball_position1 = ball_initial_position = p.getBasePositionAndOrientation(env._ball)[0]
robot_initial_position = p.getLinkState(env._robotic_arm, 0)[0]      # can be collected from optitrack
action = [0, 0, 0, 0, 0, 0]
env.step(action)
ball_position2 = p.getBasePositionAndOrientation(env._ball)[0]

table_height = 1.0
estimated_initial_velocity = estimateInitVelocity(ball_position1, ball_position2, 1/240)
estimated_hitting_point = estimate_hitting_point(ball_initial_position, robot_initial_position, estimated_initial_velocity, table_height)
print(estimated_hitting_point)

(0.4927974256909089, 2.2399837149277717)


In [8]:
import time
def ball_crossed_yz(ball_current_position, robot_current_position):#this function will detect if the ball reached Y-Z plane
    ball_current_position = p.getBasePositionAndOrientation(env._ball)[0] # can be collected from optitrack
    x_ball, y_ball, z_ball = ball_current_position
    x_robot, y_robot, z_robot = robot_current_position

    if x_ball > x_robot:  # this condition ensures the ball crossed Y-Z plane
        return True

    else:
        return False


def convert_step_to_time(step_number, frequency = 240):
    return step_number/frequency

action = [0, 0, 0, 0, 0, 0] # keep the robot stopped, we don't need it.

for step in range(5000):
    env.step(action)
    ball_current_position = p.getBasePositionAndOrientation(env._ball)[0]
    if ball_crossed_yz(ball_current_position, robot_initial_position):
        # print(convert_step_to_time(step, 240))
        break
    time.sleep(1/240) # because the simulation is at 240 Hz

real_hitting_point = (p.getBasePositionAndOrientation(env._ball)[0][1], p.getBasePositionAndOrientation(env._ball)[0][2])
print('The ball has hit the Y-Z plane at: ', real_hitting_point)

# now calculate delta

The ball has hit the Y-Z plane at:  (0.4670450215160562, 1.455905474038676)


In [9]:
## calculate delta

delta = (real_hitting_point[0] - estimated_hitting_point[0], real_hitting_point[1] - estimated_hitting_point[1])
print('delta: ', delta)

delta:  (-0.025752404174852694, -0.7840782408890956)


# Simplified with utils/physics_solvers.py

In [1]:
from TableTennisEnvironmentV0.TableTennisEnvironment import TableTennisEnv
from utils.physics_solvers import estimateInitVelocity, estimate_hitting_point, calculate_delta, had_double_bounce, had_no_bounce
import time
import pybullet as p 
import random
from tqdm import tqdm

g = 9.8
env = TableTennisEnv(show_gui=True)


pybullet build time: Feb  4 2024 12:55:26
2024-03-11 16:31:35.125 Python[11150:611898] WARNING: Secure coding is automatically enabled for restorable state! However, not on all supported macOS versions of this application. Opt-in to secure coding explicitly by implementing NSApplicationDelegate.applicationSupportsSecureRestorableState:.


## Loading the delta regressor

In [85]:
## get a new ball for the next iteration
env.get_new_ball(position=[-1.0, 0.2, 2.3])

In [11]:
import joblib
model_filename = 'delta_regressor_RF.joblib'
regressor2 = joblib.load(model_filename)
print("Model loaded successfully")

Model loaded successfully


## Estimate where the ball is gonna hit

In [86]:
import numpy as np

## Throw the ball
u_x, u_y, u_z = 5.0, 0.2, -1.5     # in meters/sec
real_initial_velocity = [u_x, u_y, u_z]
env.throw_ball(real_initial_velocity)



ball_position1 = ball_initial_position = p.getBasePositionAndOrientation(env._ball)[0]
robot_initial_position = p.getLinkState(env._robotic_arm, 0)[0]      # can be collected from optitrack
distance = robot_initial_position[0] - ball_initial_position[0]
action = [0, 0, 0, 0, 0, 0]
env.step(action)
ball_position2 = p.getBasePositionAndOrientation(env._ball)[0]

table_height = 1.0
estimated_initial_velocity = estimateInitVelocity(ball_position1, ball_position2, 1/240)
estimated_hitting_point, t1, t2 = estimate_hitting_point(ball_initial_position, robot_initial_position, estimated_initial_velocity, table_height)
input_for_delta = [[ball_initial_position[0], ball_initial_position[1], ball_initial_position[2], distance, estimated_initial_velocity[0], estimated_initial_velocity[1], estimated_initial_velocity[2], estimated_hitting_point[0], estimated_hitting_point[1]]]
print(input_for_delta)
delta = regressor2.predict(np.asarray(input_for_delta))
print('estimated hitting point (only physics)', estimated_hitting_point)
print('estimated hitting point (physics + delta)',estimated_hitting_point+delta)


[[-1.00050416, 0.1999968206, 2.29964082, 2.70050416, 4.994813347487064, 0.19979253389948548, -1.5393190042460958, 0.3080169870000016, 1.9782382088147734]]
estimated hitting point (only physics) (0.3080169870000016, 1.9782382088147734)
estimated hitting point (physics + delta) [[0.29645493 1.5267699 ]]


## Move the robot end effector to the target position

In [82]:
def get_joint_angles(robot_id, num_joints = 6):
    joint_angles = []
    for joint_index in range(2, num_joints+2):
        # Get the joint state
        joint_state = p.getJointState(robot_id, joint_index)
        # The first element of joint_state is the current position of the joint
        joint_position = joint_state[0]
        joint_angles.append(joint_position)
    return joint_angles

joint_angles1 = get_joint_angles(env._robotic_arm)

In [92]:
def calculate_joint_angles(robot_id, end_effector_pos, end_effector_orientation):
    """
    Calculates the joint angles for a given end-effector position and orientation.

    Args:
        robot_id (int): Unique ID of the robot in the PyBullet simulation.
        end_effector_pos (tuple of float): The target position of the end-effector (x, y, z).
        end_effector_orientation (tuple of float): The target orientation of the end-effector as a quaternion (x, y, z, w).

    Returns:
        list of float: The joint angles required to achieve the given end-effector position and orientation.
    """
    # Index of the end effector link
    end_effector_link_index = 8  # Assuming the last link is the end effector, adjust as needed

    # Calculate the Inverse Kinematics
    joint_angles = p.calculateInverseKinematics(
        bodyUniqueId=robot_id,
        jointDamping = [0.001,0.001,0.001,0.001,0.001,0.001, 0.001,0.001,0.001,0.001,0.001,0.001],
        endEffectorLinkIndex=end_effector_link_index,
        targetPosition=end_effector_pos,
        targetOrientation=end_effector_orientation,
        maxNumIterations=20000,
        residualThreshold=0.0001,
        solver = p.IK_DLS,
        # currentPosition = joint_angles1,
        # Additional parameters can be adjusted as needed, including joint limits, rest poses, etc.
    )

    return joint_angles


In [95]:
# Desired end-effector position and orientation (example)
estimated_goal = estimated_hitting_point + delta
end_effector_pos = (1.6, estimated_goal[0][0], estimated_goal[0][1])  # Adjust as needed
print(end_effector_pos)
end_effector_orientation = p.getQuaternionFromEuler([0, 0, 1])  # No rotation
joint_angles = calculate_joint_angles(env._robotic_arm, end_effector_pos, end_effector_orientation)
print(joint_angles)
for i, angle in enumerate(joint_angles):
    p.setJointMotorControl2(bodyIndex=env._robotic_arm,
                            jointIndex=i,
                            controlMode=p.POSITION_CONTROL,
                            targetPosition=angle)

(1.6, 0.2964549318809783, 1.5267698971855896)
(-1.9274690863555877, 1.6523723279817206, 5.107455855816456, -4.835747864918611e-06, 0.31349319712245843, 2.498255331188586, 0.9999999918548813, 0.024113493510578583, 0.9999997777174381, 1.00000019569456, 0.9999998384162034, 1.5492325087339192)


In [96]:
# Step simulation to see the result
import time
for i in range(500):
    p.stepSimulation()
    time.sleep(1./240.)

In [97]:
end_effector_link_index = 7
state = p.getLinkState(env._robotic_arm, end_effector_link_index)

# Position and orientation of the end effector
position = state[0]
orientation = state[1]

print(position)

(1.8230603997283032, 0.05488556941962593, 1.3906454340746883)


In [75]:
from math import sqrt

def calculate_distance(point1, point2):
    """
    Calculate the Euclidean distance between two 3D points.
    
    Parameters:
    - point1: A tuple of coordinates (x, y, z) for the first point.
    - point2: A tuple of coordinates (x, y, z) for the second point.
    
    Returns:
    - The Euclidean distance between point1 and point2.
    """
    return sqrt((point2[0] - point1[0])**2 + (point2[1] - point1[1])**2 + (point2[2] - point1[2])**2)

# Example usage
point1 = (1.6, 0.004906335383859935, 1.1170589108245945)  # First point coordinates
point2 = position  # Second point coordinates

# Calculate and print the distance
distance = calculate_distance(point1, point2)
print(f"The distance between {point1} and {point2} is {distance} units.")


The distance between (1.6, 0.004906335383859935, 1.1170589108245945) and (1.679902104700113, -0.17026266976963378, 1.403603472874113) is 0.34521922417809064 units.


## Check if the ball reached to the goal point

In [87]:
def ball_crossed_yz(ball_current_position, robot_current_position):#this function will detect if the ball reached Y-Z plane
    ball_current_position = p.getBasePositionAndOrientation(env._ball)[0] # can be collected from optitrack
    x_ball, y_ball, z_ball = ball_current_position
    x_robot, y_robot, z_robot = robot_current_position

    if x_ball > x_robot:  # this condition ensures the ball crossed Y-Z plane
        return True
    else:
        return False

In [88]:
action = [0, 0, 0, 0, 0, 0] # keep the robot stopped, we don't need it.

for step in range(5000):
    env.step(action)
    ball_current_position = p.getBasePositionAndOrientation(env._ball)[0]
    if ball_crossed_yz(ball_current_position, robot_initial_position):
        # print(convert_step_to_time(step, 240))
        
        break
    time.sleep(1/240) # because the simulation is at 240 Hz

real_hitting_point = (p.getBasePositionAndOrientation(env._ball)[0][1], p.getBasePositionAndOrientation(env._ball)[0][2])
print('The ball has hit the Y-Z plane at: ', real_hitting_point)

The ball has hit the Y-Z plane at:  (0.2832023936039233, 1.703394355178249)


In [ ]:
# Step simulation to see the result
import time
for i in range(1000):
    p.stepSimulation()
    time.sleep(1./240.)

In [9]:
print(calculate_delta(real_hitting_point, estimated_hitting_point))

(-0.025894158621210257, -1.1453815692902398)


In [15]:
## get a new ball for the next iteration
env.get_new_ball(position=[-1.5, -0.1, 2.3])

## Move the robot to the target position (eg: 1.7, 0.03, 0.99)

In [10]:
def calculate_joint_angles(robot_id, end_effector_pos, end_effector_orientation):
    """
    Calculates the joint angles for a given end-effector position and orientation.

    Args:
        robot_id (int): Unique ID of the robot in the PyBullet simulation.
        end_effector_pos (tuple of float): The target position of the end-effector (x, y, z).
        end_effector_orientation (tuple of float): The target orientation of the end-effector as a quaternion (x, y, z, w).

    Returns:
        list of float: The joint angles required to achieve the given end-effector position and orientation.
    """
    # Index of the end effector link
    end_effector_link_index = 8  # Assuming the last link is the end effector, adjust as needed

    # Calculate the Inverse Kinematics
    joint_angles = p.calculateInverseKinematics(
        bodyUniqueId=robot_id,
        endEffectorLinkIndex=end_effector_link_index,
        targetPosition=end_effector_pos,
        targetOrientation=end_effector_orientation,
        # Additional parameters can be adjusted as needed, including joint limits, rest poses, etc.
    )

    return joint_angles


In [11]:
# Desired end-effector position and orientation (example)
end_effector_pos = (1.7, 0.03, 0.99)  # Adjust as needed
end_effector_orientation = p.getQuaternionFromEuler([0, 0, 0])  # No rotation
joint_angles = calculate_joint_angles(env._robotic_arm, end_effector_pos, end_effector_orientation)
for i, angle in enumerate(joint_angles):
    p.setJointMotorControl2(bodyIndex=env._robotic_arm,
                            jointIndex=i,
                            controlMode=p.POSITION_CONTROL,
                            targetPosition=angle)

In [12]:
# Step simulation to see the result
import time
for i in range(1000):
    p.stepSimulation()
    time.sleep(1./240.)

In [26]:
end_effector_link_index = 8
state = p.getLinkState(env._robotic_arm, end_effector_link_index)

# Position and orientation of the end effector
position = state[0]
orientation = state[1]

print(position)

(1.3412008740122543, 0.27218671297601166, 1.458405019332481)


## set the ranges for ball initial position and initial velocity

In [7]:
ball_position_range_x = [-2, -1]
ball_position_range_y = [-0.5, 0.5]
ball_position_range_z = [1.5, 2.5]


vel_range_x = [4, 8]
vel_range_y = [-0.5, 0.5]
vel_range_z = [-3, 1]

In [8]:
def generate_random_ball_position(ball_position_range_x, ball_position_range_y, ball_position_range_z):
    x = random.uniform(ball_position_range_x[0], ball_position_range_x[1])
    y = random.uniform(ball_position_range_y[0], ball_position_range_y[1])
    z = random.uniform(ball_position_range_z[0], ball_position_range_z[1])
    return [x, y, z]

print(generate_random_ball_position(ball_position_range_x, ball_position_range_y, ball_position_range_z))

[-1.9750814865042585, 0.4977151514140895, 2.2065201720774645]


In [9]:
def generate_random_ball_velocity(vel_range_x, vel_range_y, vel_range_z):
    x = random.uniform(vel_range_x[0], vel_range_x[1])
    y = random.uniform(vel_range_y[0], vel_range_y[1])
    z = random.uniform(vel_range_z[0], vel_range_z[1])
    return [x, y, z]

In [10]:
position = generate_random_ball_position(ball_position_range_x, ball_position_range_y, ball_position_range_z)
velocity = generate_random_ball_velocity(vel_range_x, vel_range_y, vel_range_z)
env.get_new_ball(position=position)
env.throw_ball(velocity)
action = [0, 0, 0, 0, 0, 0]
for i in range(500):
    env.step(action)

## Remember, if h2 is less than table height > the ball had double bounce, ignore those samples.
## if vx*t1 > distance between ball and robot - than means the ball did not bounce, ignore those samples.
    

### Dataset Collection

In [13]:
def create_dataset_file(dataset_file = 'delta_estimate.csv'):
    with open(dataset_file, 'w') as f:
        line = 'ball_initial_pos_x'+','\
        +'ball_initial_pos_y'+','\
        +'ball_initial_pos_z'+','\
        +'distance_ball_robot'+','\
        +'estimated_initial_velocity_x'+','\
        +'estimated_initial_velocity_y'+','\
        +'estimated_initial_velocity_z'+','\
        +'estimated_hitting_point_y'+','\
        +'estimated_hitting_point_z'+','\
        +'real_hitting_point_y'+','\
        +'real_hitting_point_z'+','\
        +'delta_y'+','\
        +'delta_z'+','\
        +'\n'
        f.write(line)


create_dataset_file()

In [14]:
def write_data(ball_initial_position, 
               distance_ball_robot, 
               estimated_initial_velocity, 
               estimated_hitting_point, 
               real_hitting_point, 
               delta,
               dataset_file = 'delta_estimate.csv'):
    with open(dataset_file, 'a') as f:
        line = str(ball_initial_position[0])+','\
        +str(ball_initial_position[1])+','\
        +str(ball_initial_position[2])+','\
        +str(distance_ball_robot)+','\
        +str(estimated_initial_velocity[0])+','\
        +str(estimated_initial_velocity[1])+','\
        +str(estimated_initial_velocity[2])+','\
        +str(estimated_hitting_point[0])+','\
        +str(estimated_hitting_point[1])+','\
        +str(real_hitting_point[0])+','\
        +str(real_hitting_point[1])+','\
        +str(delta[0])+','\
        +str(delta[1])+','\
        +'\n'
        f.write(line)

In [16]:
num_experiments = 10000
table_height = 1.0
base_surface_thickness = 0.2
action = [0, 0, 0, 0, 0, 0] # no robot movement
max_steps = 1000 # it denotes how many steps to run the experiment - to avoid deadlock

for i in tqdm(range(num_experiments)):
    real_hitting_point = (-100, -100) ## out of range value
    position = generate_random_ball_position(ball_position_range_x, ball_position_range_y, ball_position_range_z)
    velocity = generate_random_ball_velocity(vel_range_x, vel_range_y, vel_range_z)
    env.get_new_ball(position=position)
    env.throw_ball(velocity)


    # do the estimations
    ball_position1 = ball_initial_position = p.getBasePositionAndOrientation(env._ball)[0]
    robot_initial_position = p.getLinkState(env._robotic_arm, 0)[0]      # can be collected from optitrack
    distance_ball_robot = robot_initial_position[0] - ball_initial_position[0]  
    env.step(action)
    ball_position2 = p.getBasePositionAndOrientation(env._ball)[0]


    estimated_initial_velocity = estimateInitVelocity(ball_position1, ball_position2, 1/240)
    estimated_hitting_point, t1, t2 = estimate_hitting_point(ball_initial_position, robot_initial_position, estimated_initial_velocity, table_height, base_surface_thickness)
    z = estimated_hitting_point[1]
    h2 = z - table_height - base_surface_thickness


    # simulate and find the real hitting position
    for step in range(max_steps):
        env.step(action)
        ball_current_position = p.getBasePositionAndOrientation(env._ball)[0]
        if ball_crossed_yz(ball_current_position, robot_initial_position):
            real_hitting_point = (p.getBasePositionAndOrientation(env._ball)[0][1], p.getBasePositionAndOrientation(env._ball)[0][2])
            delta = (real_hitting_point[0] - estimated_hitting_point[0], real_hitting_point[1] - estimated_hitting_point[1])
            if not (had_double_bounce(h2) and had_no_bounce(vx = estimated_initial_velocity[0], t1 = t1, distance_ball_robot = distance_ball_robot)):
                write_data(ball_initial_position, distance_ball_robot, estimated_initial_velocity, estimated_hitting_point, real_hitting_point, delta)
        
            
            # print(convert_step_to_time(step, 240))
            break
        # time.sleep(1/240) # because the simulation is at 240 Hz

100%|███████████████████████████████████| 10000/10000 [8:45:16<00:00,  3.15s/it]


## Training Model

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
!head delta_estimate.csv

ball_initial_pos_x,ball_initial_pos_y,ball_initial_pos_z,distance_ball_robot,estimated_initial_velocity_x,estimated_initial_velocity_y,estimated_initial_velocity_z,estimated_hitting_point_y,estimated_hitting_point_z,real_hitting_point_y,real_hitting_point_z,delta_y,delta_z,
-1.1404904348841838,0.29459225245954096,2.283251598824787,2.8404904348841837,6.7864791441764005,0.2828699774301846,-2.1568293588642007,0.41298788119964147,1.7379690943395902,0.4039223062237693,1.6454232063369119,-0.009065574975872182,-0.09254588800267838,
-1.1803215376315168,0.35090188061612154,2.4445390644436076,2.8803215376315165,5.539834933901933,0.31701861606843185,-1.675709275396784,0.515729092723847,1.9043948374447945,0.4955750601168686,1.792978935683356,-0.020154032606978456,-0.11141590176143845,
-1.2199276187579582,0.4106819463195011,1.6809033905889348,2.919927618757958,5.299524306563228,0.08771406374862067,-0.7313474136390496,0.4590105702633336,1.706518701591544,0.4489908833876091,1.0830221049556128,-0.0100

In [4]:
dataset = pd.read_csv('delta_estimate.csv')
X = dataset.iloc[:, 0:-5].values
y = dataset.iloc[:, -3:-1].values

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [6]:
print(X_test[0])
print(y_test[0])

[-1.16402148 -0.05341985  2.13085133  2.86402148  4.93314918 -0.12740935
 -2.7496837  -0.12738946  2.36556448]
[-0.04339279 -1.28334398]


In [7]:
from sklearn.tree import DecisionTreeRegressor
regressor = DecisionTreeRegressor(random_state = 0)
how_many_samples = -1
regressor.fit(X_train[:how_many_samples], y_train[:how_many_samples])

DecisionTreeRegressor(random_state=0)

In [8]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

prediction = regressor.predict(X_test)
error = r2_score(y_test[:, 1], prediction[:, 1])
print(error)

0.7021636115931904


In [9]:
from sklearn.ensemble import RandomForestRegressor
how_many_samples = -1
regressor2 = RandomForestRegressor(n_estimators=20)
regressor2.fit(X_train[:how_many_samples], y_train[:how_many_samples])


RandomForestRegressor(n_estimators=20)

In [10]:
from sklearn.metrics import mean_squared_error

prediction = regressor2.predict(X_test)
error = r2_score(y_test[:, 1], prediction[:, 1])
print(error)

0.8518265285660642


In [11]:
print(y_test[0])
print(prediction[0])

[-0.04339279 -1.28334398]
[-0.08677592 -1.27997484]


# Save the model

In [30]:
import joblib
model_filename = 'delta_regressor_RF.joblib'
joblib.dump(regressor2, model_filename)

['delta_regressor_RF.joblib']